## 1. Teste Z para diferença entre duas proporções (A/B test)

**O que é:** testa se a diferença entre duas taxas de conversão (proporções) é estatisticamente significativa, ou se pode ser só ruído amostral.

**Quando usar:** dados binários (converteu / não converteu) em dois grupos independentes (A e B), com amostra grande o suficiente pra aproximação normal (regra prática: `n*p` e `n*(1-p)` ≥ 5 em cada grupo).

**Hipóteses:**
- H0: `p_A = p_B` (não há diferença real de conversão)
- H1: `p_A ≠ p_B` (teste bicaudal, padrão do `proportions_ztest`)

**Estatística:**

$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}$$

onde `p̂` é a proporção combinada (pooled) das duas amostras.

**Código abaixo:** calcula conversão de A e B a partir de vetores binários, a diferença absoluta, o lift relativo, e roda o teste z retornando o p-valor.

In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Dados (amostra maior: 200 visitantes por grupo, gerados por simulacao)
np.random.seed(42)
A = np.random.binomial(1, 0.30, 200)  # grupo A: conversao real ~30%
B = np.random.binomial(1, 0.50, 200)  # grupo B: conversao real ~50%

# Conversion rates
conversion_A = A.mean()
conversion_B = B.mean()

# Difference
difference = conversion_B - conversion_A

# Relative lift
lift = difference / conversion_A

# Hypothesis test
successes = np.array([A.sum(), B.sum()])
nobs = np.array([len(A), len(B)])

z_stat, p_value = proportions_ztest(
    successes,
    nobs
)

print("Conversion A:", conversion_A)
print("Conversion B:", conversion_B)
print("Difference:", difference)
print("Lift:", lift)
print("p-value:", p_value)

Conversion A: 0.3
Conversion B: 0.545
Difference: 0.24500000000000005
Lift: 0.8166666666666669
p-value: 7.051366865845907e-07


## 2. Decisão estatística (regra do p-valor)

**O que é:** compara o p-valor obtido no teste anterior com o nível de significância `alpha` (aqui 5%) pra decidir se rejeita ou não H0.

**Regra:**
- `p-value < alpha` → rejeita H0 → diferença estatisticamente significativa
- `p-value ≥ alpha` → não rejeita H0 → não há evidência suficiente de diferença

**Cuidado de interpretação:** "não significativo" não prova que as proporções são iguais, só que a amostra não teve poder suficiente pra detectar a diferença com confiança.

**Código abaixo:** aplica essa regra ao `p_value` calculado na célula anterior.

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Statistically significant difference


## 3. Teste Z de proporções a partir de contagens agregadas

**O que é:** mesmo teste do bloco 1 (teste z para duas proporções), mas agora os dados já vêm agregados como contagem de sucessos e total de observações por grupo, em vez de vetores binários individuais — útil quando você só tem os totais (ex: relatório de campanha).

**Hipóteses:** as mesmas do teste anterior (H0: proporções iguais vs H1: proporções diferentes).

**Interpretação do resultado:** aqui `z` é negativo porque a segunda proporção (70/1000 = 7%) é maior que a primeira (50/1000 = 5%), então a diferença tem sinal invertido. O p-valor (~0,06) fica levemente acima do `alpha = 0,05` comum, ou seja, no limiar da significância.

**Código abaixo:** roda `proportions_ztest` diretamente com listas de sucessos e observações.

In [4]:
from statsmodels.stats.proportion import proportions_ztest

sucessos = [50, 70]
observacoes = [1000, 1000]

z, p = proportions_ztest(sucessos, observacoes)

print(z)
print(p)

-1.883108942886774
0.05968560553242621


## 4. Teste t de Student para amostras independentes

**O que é:** compara as médias de dois grupos **independentes** (sujeitos diferentes em A e B) pra ver se a diferença entre elas é estatisticamente significativa.

**Quando usar:** variável contínua, dois grupos não pareados, aproximadamente normal (ou `n` grande o suficiente pelo Teorema Central do Limite). Por padrão o `ttest_ind` do scipy assume variâncias iguais entre os grupos (teste t de Student clássico); se as variâncias forem muito diferentes, o certo é usar `equal_var=False` (teste de Welch).

**Hipóteses:**
- H0: `média_A = média_B`
- H1: `média_A ≠ média_B`

**Código abaixo:** compara os grupos A e B (ex: dois métodos, dois tratamentos) e retorna o p-valor da diferença de médias.

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

# Amostra maior: 30 observacoes por grupo
np.random.seed(1)
A = np.random.normal(90, 15, 30).round(1)
B = np.random.normal(105, 15, 30).round(1)

t, p = ttest_ind(A, B)

print(p)

1.7493992371161784e-05


## 5. Teste t pareado (amostras dependentes)

**O que é:** compara duas medições feitas **no mesmo indivíduo/unidade**, em dois momentos (antes/depois) ou duas condições — diferente do teste independente, aqui cada linha de "antes" tem seu par exato em "depois".

**Por que usar pareado em vez de independente:** ao comparar a mesma unidade contra ela mesma, você remove a variação individual (cada pessoa é seu próprio controle), o que aumenta o poder do teste pra detectar diferenças reais.

**Hipóteses:**
- H0: a diferença média entre pares é zero (`média(depois - antes) = 0`)
- H1: a diferença média entre pares é diferente de zero

**Código abaixo:** compara `antes` vs `depois` (ex: performance pré/pós intervenção) usando `ttest_rel`.

In [ ]:
from scipy.stats import ttest_rel
import numpy as np

# Amostra maior: 25 pares antes/depois
np.random.seed(7)
antes = np.random.normal(11, 3, 25).round(1)
melhora = np.random.normal(2, 1.5, 25).round(1)
depois = (antes + melhora).round(1)

t, p = ttest_rel(antes, depois)

print(p)

0.0001869487103792182


## 6. Teste Qui-quadrado de independência

**O que é:** testa se existe associação entre **duas variáveis categóricas**, organizadas em uma tabela de contingência (linhas x colunas de contagens).

**Quando usar:** dados de contagem/frequência (não médias), ex: "grupo (controle/tratamento) x converteu (sim/não)".

**Hipóteses:**
- H0: as variáveis são independentes (não há associação)
- H1: as variáveis são dependentes (há associação)

**Como funciona:** compara as frequências observadas com as frequências esperadas caso as variáveis fossem independentes; quanto maior a distância entre observado e esperado, maior a estatística qui-quadrado e menor o p-valor.

**Código abaixo:** roda `chi2_contingency` numa tabela 2x2 de conversões (ex: grupo x converteu/não converteu).

In [7]:
from scipy.stats import chi2_contingency

tabela = [
    [120, 880],
    [150, 850]
]

chi2, p, dof, expected = chi2_contingency(tabela)

print(p)

0.057746841509689


## 7. ANOVA de um fator (F-test)

**O que é:** compara as médias de **três ou mais grupos** simultaneamente, testando se pelo menos um deles difere dos demais.

**Por que não usar vários t-tests em pares:** cada teste par a par tem chance de erro Tipo I (falso positivo); ao rodar vários testes o erro acumulado (inflação do alpha) cresce. A ANOVA testa todos os grupos de uma vez, controlando esse problema.

**Hipóteses:**
- H0: todas as médias são iguais (`média_A = média_B = média_C`)
- H1: pelo menos uma média é diferente das outras

**Limitação:** a ANOVA diz que existe diferença, mas não diz **qual** grupo difere — pra isso seria necessário um teste post-hoc (ex: Tukey HSD).

**Código abaixo:** compara três grupos (A, B, C) com `f_oneway` e retorna o p-valor do teste F.

In [ ]:
from scipy.stats import f_oneway
import numpy as np

# Amostra maior: 30 observacoes por grupo
np.random.seed(3)
A = np.random.normal(88, 8, 30).round(1)
B = np.random.normal(104, 8, 30).round(1)
C = np.random.normal(76, 8, 30).round(1)

F, p = f_oneway(A, B, C)

print(p)

4.555858799454035e-17


## 8. Teste de Mann-Whitney U (não paramétrico)

**O que é:** equivalente não-paramétrico do teste t independente — compara se as distribuições de dois grupos independentes são diferentes, sem assumir normalidade dos dados.

**Quando usar:** quando os dados não são normais, quando a amostra é pequena, ou quando há outliers fortes que distorceriam a média (repare que o grupo A tem um valor de 25000 destoante dos demais — a média seria fortemente puxada por ele, mas o teste baseado em **ranks/medianas** é robusto a isso).

**Hipóteses:**
- H0: as duas distribuições são iguais (nenhuma tende a ter valores maiores que a outra)
- H1: uma distribuição tende a ter valores sistematicamente maiores que a outra

**Como funciona:** ao invés de comparar médias, o teste converte os valores em postos (ranks) combinados dos dois grupos e compara a soma dos ranks entre eles.

**Código abaixo:** compara os grupos A e B (com outlier em A) usando `mannwhitneyu`.

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np

# Amostra maior: 30 observacoes por grupo (mantendo um outlier no grupo A)
np.random.seed(5)
A = np.random.normal(3500, 300, 29).round(0).tolist() + [25000]  # outlier
B = np.random.normal(3600, 300, 30).round(0).tolist()

u, p = mannwhitneyu(A, B)

print(p)

0.010092817132935963


## 9. Correlação de Pearson

**O que é:** mede a força e a direção da associação **linear** entre duas variáveis contínuas (ex: horas de estudo x nota).

**Quando usar:** ambas variáveis contínuas, relação esperada aproximadamente linear (não captura relações não-lineares).

**Interpretação do coeficiente `r`:**
- `r` varia de -1 a +1
- próximo de +1 → forte correlação positiva (uma sobe, a outra sobe)
- próximo de -1 → forte correlação negativa
- próximo de 0 → pouca ou nenhuma correlação linear

**Hipóteses do p-valor associado:**
- H0: `r = 0` (não há correlação linear na população)
- H1: `r ≠ 0`

**Cuidado:** correlação não implica causalidade — `r` alto só indica associação linear, não que uma variável causa a outra.

**Código abaixo:** calcula `r` e o p-valor entre `horas` de estudo e `notas` com `pearsonr`.

In [ ]:
from scipy.stats import pearsonr
import numpy as np

# Amostra maior: 20 observacoes
np.random.seed(9)
horas = np.arange(1, 21)
notas = (50 + 2.0*horas + np.random.normal(0, 5, 20)).round(1)

r, p = pearsonr(horas, notas)

print(r)
print(p)

0.9614001751341554
1.5702616196622007e-11


## 10. Tamanho de amostra para estimar uma média (margem de erro conhecida)

**O que é:** calcula quantas observações são necessárias pra estimar uma média populacional dentro de uma margem de erro desejada, com um nível de confiança escolhido — pensado **antes** de coletar os dados (planejamento amostral), diferente dos testes anteriores que analisavam dados já coletados.

**Quando usar:** quando você sabe (ou tem uma estimativa de) o desvio-padrão da variável (`sigma`), e quer definir de antemão o `n` mínimo do estudo/experimento.

**Fórmula:**

$$n = \left(\frac{Z \cdot \sigma}{E}\right)^2$$

onde `Z` é o valor crítico da normal padrão para o nível de confiança (ex: 1,96 para 95%), `σ` é o desvio-padrão estimado, e `E` é a margem de erro máxima que você aceita.

**Código abaixo:** com 95% de confiança, `sigma=10` e margem de erro de 2, calcula o `n` mínimo necessário.

In [1]:
from scipy.stats import norm
import math

# Parâmetros
confidence = 0.95
sigma = 10
margin_error = 2

# Valor crítico Z
Z = norm.ppf((1 + confidence) / 2)

# Cálculo
n = (Z * sigma / margin_error) ** 2

print(f"Valor de Z: {Z:.3f}")
print(f"Tamanho da amostra: {math.ceil(n)}")

Valor de Z: 1.960
Tamanho da amostra: 97


## 11. A mesma fórmula, encapsulada em função reutilizável

**O que é:** reaproveita a fórmula da célula anterior, agora dentro de uma função `sample_size_mean(confidence, sigma, margin_error)` — útil pra recalcular o tamanho de amostra rapidamente com parâmetros diferentes, sem duplicar código.

**Por que isso importa na prática:** em um projeto real você normalmente testa vários cenários (margens de erro ou desvios-padrão diferentes); ter isso como função evita copiar e colar a fórmula toda vez.

**Nota:** os parâmetros usados (`confidence=0.95, sigma=10, margin_error=2`) são os mesmos da célula anterior, por isso o resultado é idêntico (n=97).

**Código abaixo:** define a função e chama com os mesmos valores de antes, só que agora de forma reutilizável.

In [3]:
from scipy.stats import norm
import math

def sample_size_mean(confidence, sigma, margin_error):
    Z = norm.ppf((1 + confidence) / 2)
    n = (Z * sigma / margin_error) ** 2
    return math.ceil(n)

sample_size_mean(0.95, 10, 2)

97

## 12. Tamanho de amostra para estimar uma proporção

**O que é:** versão da fórmula de planejamento amostral para quando a variável de interesse é uma **proporção** (ex: taxa de conversão, taxa de aprovação, taxa de churn) em vez de uma média contínua.

**Fórmula:**

$$n = \frac{Z^2 \cdot p(1-p)}{E^2}$$

onde `p` é a proporção esperada (quando não se sabe, usa-se `p=0.5`, que maximiza `p(1-p)` e dá o tamanho de amostra mais conservador/seguro) e `E` é a margem de erro desejada.

**Quando usar:** planejar pesquisas de opinião, dimensionar quantos usuários observar pra estimar uma taxa de conversão com certa precisão, etc.

**Código abaixo:** com 95% de confiança, `p=0.40` esperado e margem de erro de 5 pontos percentuais, calcula o `n` mínimo necessário (369).

In [2]:
from scipy.stats import norm
import math

confidence = 0.95
p = 0.40
margin_error = 0.05

Z = norm.ppf((1 + confidence) / 2)

n = (Z**2 * p * (1 - p)) / (margin_error**2)

print(f"Tamanho da amostra: {math.ceil(n)}")

Tamanho da amostra: 369


## 13. Poder estatístico e tamanho de amostra para um teste A/B

**O que é:** diferente das duas células anteriores (que estimam uma proporção/média isolada), aqui calculamos quantas observações **por grupo** são necessárias pra conseguir **detectar** uma diferença específica entre duas proporções, dado um poder estatístico desejado — ou seja, planejamento de experimento (A/B test) antes de rodar.

**Conceitos-chave:**
- **Effect size (Cohen's h):** mede o "tamanho" da diferença entre as duas proporções numa escala padronizada, calculado aqui por `proportion_effectsize(p1, p2)`.
- **Power (poder do teste) = 1 − β:** probabilidade de detectar a diferença quando ela realmente existe (evitar o erro Tipo II — não detectar um efeito real). Aqui usa-se o padrão de mercado de 80%.
- **Alpha:** taxa de erro Tipo I aceita (falso positivo), aqui 5%.

**Cenário do código:** comparar uma conversão de 10% (p1) contra 12% (p2) — uma diferença pequena — com poder de 80% e alpha de 5%.

**Por que o `n` sai tão grande (≈3835 por grupo):** diferenças pequenas entre proporções exigem amostras muito maiores pra serem detectadas com confiança — conecta diretamente com o teste Z de proporções do início do notebook: com pouca amostra (como os primeiros exemplos), mesmo diferenças reais podem não aparecer como "significativas".

In [4]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Taxas de conversão
p1 = 0.10
p2 = 0.12

# Calcula o tamanho do efeito (Cohen's h)
effect_size = proportion_effectsize(p1, p2)

analysis = NormalIndPower()

n = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    ratio=1.0
)

print(f"Amostra por grupo: {n:.0f}")

Amostra por grupo: 3835
